### *ETAPA 6 Preparación del modelo analítico para Power BI*

En esta etapa se consolidan los resultados obtenidos durante el análisis en un único archivo Excel estructurado para su consumo desde Power BI. El objetivo es centralizar la información necesaria para construir el tablero ejecutivo solicitado por el cliente.

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

In [21]:
# ==========================================
# Cargar dataset
# ==========================================

segmentacion = pd.read_excel(
    "../data/processed/segmentacion_final.xlsx"
)

transacciones = pd.read_excel(
    "../data/raw/DataTransacciones_PruebasIngreso.xlsx",
    sheet_name="transacciones"
)

In [22]:
# ==========================================
# Construcción KPIs
# ==========================================

ventas_totales = segmentacion["Monto ventas Acumulado"].sum()

comision_total = ventas_totales * 0.05

ventas_ultimo_mes = segmentacion["Monto ventas ultimo mes"].sum()

comision_ultimo_mes = ventas_ultimo_mes * 0.05

comercios_activos = (
    segmentacion["Monto ventas ultimo mes"] > 0
).sum()

ticket_promedio = (
    ventas_totales /
    segmentacion["frecuencia"].sum()
)

frecuencia_promedio = (
    segmentacion["frecuencia"].mean()
)

In [23]:
# ==========================================
# Tabla resumen KPIs
# ==========================================

kpis_generales = pd.DataFrame({

    "KPI":[

        "Ventas Totales",
        "Comisión Estimada",
        "Ventas Último Mes",
        "Comisión Último Mes",
        "Comercios Activos",
        "Ticket Promedio",
        "Frecuencia Promedio"

    ],

    "Valor":[

        ventas_totales,
        comision_total,
        ventas_ultimo_mes,
        comision_ultimo_mes,
        comercios_activos,
        ticket_promedio,
        frecuencia_promedio

    ]

})

In [24]:
# ==========================================
# KPIs por segmento
# ==========================================

dashboard_segmentos = (

    segmentacion

    .groupby("segmento")

    .agg(

        comercios=("Id_comercio","count"),

        transacciones=("frecuencia","sum"),

        ventas_acumuladas=("Monto ventas Acumulado","sum"),

        ventas_ultimo_mes=("Monto ventas ultimo mes","sum")

    )

)

In [25]:
dashboard_segmentos["comision_acumulada"] = (

    dashboard_segmentos["ventas_acumuladas"]*0.05

)

dashboard_segmentos["comision_ultimo_mes"]=(

    dashboard_segmentos["ventas_ultimo_mes"]*0.05

)

dashboard_segmentos["% ventas"]=(

    dashboard_segmentos["ventas_acumuladas"]

    /

    dashboard_segmentos["ventas_acumuladas"].sum()

)*100

dashboard_segmentos["% comisión"]=(

    dashboard_segmentos["comision_acumulada"]

    /

    dashboard_segmentos["comision_acumulada"].sum()

)*100

In [26]:
historico = (

    transacciones

    .groupby("mes-anio transaccion")

    .agg(

        transacciones=("cantidad transacciones","sum"),

        comercios_activos=("Id_comercio","nunique")

    )

    .reset_index()

)

In [27]:
# ==========================================
# Exportar archivo final para Power BI
# ==========================================

from pathlib import Path

# Crear carpeta dashboard si no existe
Path("../data/dashboard").mkdir(
    parents=True,
    exist_ok=True
)

# Exportar un único archivo Excel con todas las hojas
with pd.ExcelWriter(
    "../data/dashboard/dashboard_prueba.xlsx",
    engine="openpyxl"
) as writer:

    # ==========================
    # Hoja 1 - KPIs generales
    # ==========================
    kpis.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )

    # ==========================
    # Hoja 2 - Resumen segmentos
    # ==========================
    segmentos.to_excel(
        writer,
        sheet_name="Segmentos"
    )

    # ==========================
    # Hoja 3 - Comercios
    # ==========================
    segmentacion.to_excel(
        writer,
        sheet_name="Comercios",
        index=False
    )

print("✅ dashboard_prueba.xlsx exportado correctamente.")

✅ dashboard_prueba.xlsx exportado correctamente.
